In [11]:
# , 1686, 4221, 1801, 15078
import requests
import json
import time
import pandas as pd
import random
import os
import concurrent.futures
import threading
from bs4 import BeautifulSoup

# ================= CẤU HÌNH =================
PARENT_CATEGORIES = [931, 1686, 4221, 1801, 15078]     # danh mục cha (có thể thêm nhiều)
PAGES_PER_SUB_CAT = 3         # số trang mỗi sub-cat
OUTPUT_FILE = "tiki_Chi_3_pages.csv"

# Giảm thread xuống 4-5 như yêu cầu
MAX_WORKERS = 4

# Batch size để ghi CSV 1 lần
CSV_BATCH_SIZE = 200

# Throttling: khoảng delay tối thiểu giữa 2 request của 1 worker
# (giữ an toàn, tăng nếu bị 429)
MIN_DELAY = 0.25   # seconds
MAX_DELAY = 0.8    # seconds

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Referer': 'https://tiki.vn/',
    'Origin': 'https://tiki.vn'
}

REQUIRED_COLUMNS = [
    'product_id', 'product_name', 'product_url',
    'category_id', 'category_name', 'category_root_name',
    'price', 'original_price', 'discount_rate',
    'quantity_sold', 'rating_average', 'review_count',
    'is_return_policy', 'is_freeship_xtra', 'is_authentic',
    'image_count', 'video_count',
    'is_brand', 'brand_name', 'origin',
    'store_id', 'store_name', 'store_review_count', 'total_follower', 'is_official',
    'cancel_by_seller_rate', 'cancel_by_seller_rate_status',
    'return_rate', 'return_rate_status'
]

# ================= GLOBALS & LOCKS =================
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

SHOP_CACHE = {}           # cache shop info by seller_id
RESULTS_BATCH = []        # batch buffer
BATCH_LOCK = threading.Lock()
CSV_LOCK = threading.Lock()
PRINT_LOCK = threading.Lock()

# ================= Helper: safe_request using session =================
def safe_request(url, params=None, max_retries=3, timeout=10):
    """
    Use SESSION.get with retry/backoff. Returns Response or None.
    """
    backoff = 1.0
    for attempt in range(max_retries):
        try:
            resp = SESSION.get(url, params=params, timeout=timeout)
            if resp.status_code == 200:
                return resp
            if resp.status_code == 404:
                return None
            # if rate limited or server error -> retry with backoff
            time.sleep(backoff + random.random()*0.4)
            backoff *= 1.8
        except requests.RequestException:
            time.sleep(backoff + random.random()*0.4)
            backoff *= 1.8
    return None

# ================= Shop info helpers (widget + ovl + fallback scrape) =================
def get_seller_widget_info(seller_id, mpid, spid):
    """
    Try product-detail widget endpoint for follower/review.
    Return dict { total_follower, store_review_count } (values maybe None)
    """
    if not seller_id:
        return {'total_follower': None, 'store_review_count': None}
    url = "https://api.tiki.vn/product-detail/v2/widgets/seller"
    params = {'seller_id': seller_id, 'mpid': mpid, 'spid': spid, 'platform': 'desktop', 'version': 3}
    resp = safe_request(url, params=params)
    if resp:
        try:
            d = resp.json().get('data', {}).get('seller', {})
            return {
                'total_follower': d.get('total_follower'),
                'store_review_count': d.get('review_count')
            }
        except Exception:
            pass
    return {'total_follower': None, 'store_review_count': None}

def get_seller_ovl_stats(seller_id):
    """
    Try seller OVl API for cancel/return rates
    Caches result in SHOP_CACHE.
    """
    if not seller_id:
        return {'cancel_rate': None, 'cancel_status': None, 'return_rate': None, 'return_status': None}
    if seller_id in SHOP_CACHE and SHOP_CACHE[seller_id].get('ovl_checked'):
        cached = SHOP_CACHE[seller_id]
        return {
            'cancel_rate': cached.get('cancel_rate'),
            'cancel_status': cached.get('cancel_status'),
            'return_rate': cached.get('return_rate'),
            'return_status': cached.get('return_status')
        }

    url = f"https://seller-store-api.tiki.vn/ovl-performances/{seller_id}"
    resp = safe_request(url)
    info = {'cancel_rate': None, 'cancel_status': None, 'return_rate': None, 'return_status': None}
    if resp:
        try:
            d = resp.json()
            info['cancel_rate'] = d.get('cancel_by_seller_rate_l4w')
            info['cancel_status'] = d.get('cancel_by_seller_rate_l4w_status')
            info['return_rate'] = d.get('return_rate_l4w')
            info['return_status'] = d.get('return_rate_l4w_status')
            # normalize percent to string with percent sign if numeric
            if info['cancel_rate'] is not None:
                try:
                    info['cancel_rate'] = f"{float(info['cancel_rate'])}%"
                except:
                    pass
            if info['return_rate'] is not None:
                try:
                    info['return_rate'] = f"{float(info['return_rate'])}%"
                except:
                    pass
        except Exception:
            pass

    # store partial in cache
    SHOP_CACHE.setdefault(seller_id, {}).update(info)
    SHOP_CACHE[seller_id]['ovl_checked'] = True
    return info

def scrape_shop_html_by_slug(slug):
    """
    Fallback: scrape shop HTML to get follower/review and performance statuses.
    """
    if not slug:
        return {'total_follower': None, 'store_review_count': None, 'cancel_status': None, 'return_status': None}
    url = f"https://tiki.vn/cua-hang/{slug}?platform=web"
    resp = safe_request(url)
    out = {'total_follower': None, 'store_review_count': None, 'cancel_status': None, 'return_status': None}
    if not resp:
        return out
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
        # follower
        ftag = soup.select_one(".seller-info__follow strong")
        if ftag:
            out['total_follower'] = ftag.text.strip().replace('.', '').replace(',', '')
        # review count
        rtag = soup.select_one(".seller-info__rating span")
        if rtag:
            txt = rtag.text.strip().split()[0] if rtag.text.strip() else None
            if txt:
                out['store_review_count'] = txt.replace('.', '').replace(',', '')
        # performance items
        items = soup.select(".seller-performance__item")
        for it in items:
            title_tag = it.select_one(".seller-performance__item-title")
            value_tag = it.select_one(".seller-performance__item-value")
            if not title_tag or not value_tag:
                continue
            t = title_tag.text.strip().lower()
            v = value_tag.text.strip()
            if "tỷ lệ hủy" in t:
                out['cancel_status'] = v
            if "tỷ lệ hoàn" in t:
                out['return_status'] = v
    except Exception:
        pass
    return out

def get_combined_shop_info(seller_obj_from_product, mpid=None, spid=None):
    """
    Try several sources in order:
     1) widget (fast)
     2) ovl api (for rates)
     3) scrape html (fallback)
    Return dict containing required fields.
    """
    result = {
        'total_follower': None,
        'store_review_count': None,
        'cancel_by_seller_rate': None,
        'cancel_by_seller_rate_status': None,
        'return_rate': None,
        'return_rate_status': None
    }
    seller_id = None
    seller_slug = None
    if seller_obj_from_product:
        seller_id = seller_obj_from_product.get('id') or seller_obj_from_product.get('store_id')
        seller_slug = seller_obj_from_product.get('slug') or seller_obj_from_product.get('url_key')
        # product JSON may already have review_count
        if seller_obj_from_product.get('review_count'):
            result['store_review_count'] = seller_obj_from_product.get('review_count')

    # 1) widget
    w = get_seller_widget_info(seller_id, mpid, spid)
    if w.get('total_follower') is not None:
        result['total_follower'] = w.get('total_follower')
    if w.get('store_review_count') is not None:
        result['store_review_count'] = w.get('store_review_count')

    # 2) ovl stats
    ovl = get_seller_ovl_stats(seller_id)
    if ovl.get('cancel_rate') is not None:
        result['cancel_by_seller_rate'] = ovl.get('cancel_rate')
    if ovl.get('cancel_status') is not None:
        result['cancel_by_seller_rate_status'] = ovl.get('cancel_status')
    if ovl.get('return_rate') is not None:
        result['return_rate'] = ovl.get('return_rate')
    if ovl.get('return_status') is not None:
        result['return_rate_status'] = ovl.get('return_status')

    # 3) if any of follower/review/status missing -> scrape HTML
    if (result['total_follower'] is None or result['store_review_count'] is None or
        result['cancel_by_seller_rate_status'] is None or result['return_rate_status'] is None):
        s = scrape_shop_html_by_slug(seller_slug)
        if result['total_follower'] is None and s.get('total_follower') is not None:
            result['total_follower'] = s.get('total_follower')
        if result['store_review_count'] is None and s.get('store_review_count') is not None:
            result['store_review_count'] = s.get('store_review_count')
        if result['cancel_by_seller_rate_status'] is None and s.get('cancel_status') is not None:
            result['cancel_by_seller_rate_status'] = s.get('cancel_status')
        if result['return_rate_status'] is None and s.get('return_status') is not None:
            result['return_rate_status'] = s.get('return_status')

    # keep cache quick-access
    if seller_id:
        SHOP_CACHE.setdefault(seller_id, {}).update(result)

    return result

# ================= Listing & Detail =================
def get_sub_categories(parent_id):
    url = "https://tiki.vn/api/personalish/v1/blocks/listings"
    params = {'limit': 40, 'include': 'advertisement', 'aggregations': 2, 'version': 'home-persionalized', 'category': parent_id, 'page': 1}
    try:
        cat_info = safe_request(f"https://tiki.vn/api/v2/categories/{parent_id}")
        root_name = parent_id
        if cat_info:
            try:
                root_name = cat_info.json().get('name') or str(parent_id)
            except:
                root_name = str(parent_id)
        resp = safe_request(url, params=params)
        if not resp:
            return [], root_name
        filters = resp.json().get('filters', [])
        sub_cats = []
        for f in filters:
            if f.get('query_name') in ['category', 'tiki_category']:
                for item in f.get('values', []):
                    cid = item.get('query_value') or item.get('url_key')
                    if cid and isinstance(cid, str) and cid.startswith('c') and cid[1:].isdigit():
                        cid = cid[1:]
                    if cid and str(cid) != str(parent_id):
                        sub_cats.append({'id': int(cid) if str(cid).isdigit() else cid, 'name': item.get('display_value')})
                break
        return sub_cats, root_name
    except Exception:
        return [], str(parent_id)

def get_listing_items(category_id, page):
    url = "https://tiki.vn/api/personalish/v1/blocks/listings"
    params = {'limit': 40, 'include': 'advertisement', 'aggregations': 2, 'version': 'home-persionalized', 'category': category_id, 'page': page}
    resp = safe_request(url, params=params)
    if not resp:
        return []
    try:
        data = resp.json().get('data', [])
        items = []
        for x in data:
            items.append({
                'id': x.get('id'),
                'lp': x.get('price'),
                'lop': x.get('original_price'),
                'ldr': x.get('discount_rate')
            })
        return items
    except Exception:
        return []

def get_product_detail(listing_item, root_cat_name):
    """
    Core detail logic using SESSION + combined shop info fallback.
    """
    spid = listing_item['id']
    url = f"https://tiki.vn/api/v2/products/{spid}"
    resp = safe_request(url)
    # Always sleep a little to avoid burst even with session reuse
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
    if not resp:
        return None
    try:
        data = resp.json()
    except Exception:
        return None

    # price fallback to listing values
    price = listing_item.get('lp') or data.get('price')
    original_price = listing_item.get('lop') or data.get('original_price')
    discount_rate = listing_item.get('ldr') or data.get('discount_rate')
    if (discount_rate == 0 or discount_rate is None) and original_price and price and original_price > price:
        try:
            discount_rate = int(round((1 - (price / original_price)) * 100))
        except:
            pass

    seller = data.get('current_seller', {}) or {}
    seller_id = seller.get('id') or seller.get('store_id')
    mpid = data.get('master_id') or spid

    # get shop info combination (widget + ovl + html)
    shop_info = get_combined_shop_info(seller, mpid=mpid, spid=spid)

    # breadcrumbs
    cats = data.get('breadcrumbs', [])
    sub_cat = cats[-1] if cats else {}

    # qty sold variations
    qty_sold = data.get('all_time_quantity_sold')
    if not qty_sold:
        try:
            qty_sold = data.get('quantity_sold', {}).get('value')
        except:
            qty_sold = None

    # freeship
    is_freeship = 0
    for b in data.get('badges_new', []):
        if b.get('code') == 'tiki_free_shipping_reward':
            is_freeship = 1
            break

    # origin: search across all spec groups
    origin = None
    for spec in data.get('specifications', []):
        for attr in spec.get('attributes', []):
            if attr.get('code') == 'origin':
                origin = attr.get('value')
                break
        if origin:
            break

    brand_name = data.get('brand', {}).get('name')
    is_brand = 1 if (brand_name and brand_name.lower().strip() != 'oem') else 0

    item = {
        'product_id': data.get('id'),
        'product_name': data.get('name'),
        'product_url': f"https://tiki.vn/{data.get('url_path')}",
        'category_id': sub_cat.get('category_id'),
        'category_name': sub_cat.get('name'),
        'category_root_name': root_cat_name,
        'price': price,
        'original_price': original_price,
        'discount_rate': discount_rate,
        'quantity_sold': qty_sold,
        'rating_average': data.get('rating_average'),
        'review_count': data.get('review_count'),
        'is_return_policy': 1 if data.get('return_policy') else 0,
        'is_freeship_xtra': is_freeship,
        'is_authentic': 1 if data.get('is_authentic') else 0,
        'image_count': len(data.get('images', [])) if data.get('images') else 0,
        'video_count': len(data.get('video_url', [])) if data.get('video_url') else 0,
        'is_brand': is_brand,
        'brand_name': brand_name,
        'origin': origin,
        'seller_id': seller_id,
        'store_id': seller.get('store_id'),
        'store_name': seller.get('name'),
        'store_review_count': shop_info.get('store_review_count'),
        'total_follower': shop_info.get('total_follower'),
        'is_official': 1 if seller.get('is_official_store') else 0,
        'cancel_by_seller_rate': shop_info.get('cancel_by_seller_rate'),
        'cancel_by_seller_rate_status': shop_info.get('cancel_by_seller_rate_status'),
        'return_rate': shop_info.get('return_rate'),
        'return_rate_status': shop_info.get('return_rate_status')
    }
    return item

# ================= Batch write helpers =================
def push_result(detail):
    """
    Append to RESULTS_BATCH, if reach CSV_BATCH_SIZE then flush to file.
    Thread-safe.
    """
    if not detail:
        return
    with BATCH_LOCK:
        RESULTS_BATCH.append(detail)
        if len(RESULTS_BATCH) >= CSV_BATCH_SIZE:
            flush_batch_to_csv()

def flush_batch_to_csv(force=False):
    """
    Write RESULTS_BATCH to CSV. If force True, write remaining even if < batch size.
    """
    with BATCH_LOCK:
        if not RESULTS_BATCH:
            return
        # create DataFrame once
        df = pd.DataFrame(RESULTS_BATCH)
        # ensure columns
        df = df.reindex(columns=REQUIRED_COLUMNS)
        # thread-safe file write
        with CSV_LOCK:
            header = False
            if not os.path.exists(OUTPUT_FILE) or os.path.getsize(OUTPUT_FILE) == 0:
                header = True
            df.to_csv(OUTPUT_FILE, mode='a', header=header, index=False, encoding='utf-8-sig')
        RESULTS_BATCH.clear()

# ================= Worker wrapper =================
def process_task(task):
    listing_item, root_name = task
    try:
        detail = get_product_detail(listing_item, root_name)
        push_result(detail)
        # small random sleep to reduce burstiness
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
        # visual indicator
        with PRINT_LOCK:
            if detail and detail.get('total_follower'):
                print('+', end='', flush=True)
            elif detail:
                print('.', end='', flush=True)
            else:
                print('x', end='', flush=True)
        return True
    except Exception:
        with PRINT_LOCK:
            print('e', end='', flush=True)
        return False

# ================= MAIN =================
def main():
    # ensure output file header if not exist
    if not os.path.exists(OUTPUT_FILE):
        pd.DataFrame(columns=REQUIRED_COLUMNS).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

    total_saved = 0
    try:
        for parent in PARENT_CATEGORIES:
            sub_cats, root_name = get_sub_categories(parent)
            print(f"\nCrawling parent {parent} -> {root_name} | subcats: {len(sub_cats)}")
            for sub in sub_cats:
                print(f"\n Sub: {sub.get('name')} (ID {sub.get('id')})")
                for page in range(1, PAGES_PER_SUB_CAT + 1):
                    items = get_listing_items(sub['id'], page)
                    if not items:
                        print(f"[SKIP Pg{page}]", end=' ')
                        time.sleep(1.0)
                        continue
                    tasks = [(it, root_name) for it in items]
                    # ThreadPool with limited workers
                    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                        futures = [executor.submit(process_task, t) for t in tasks]
                        for fut in concurrent.futures.as_completed(futures):
                            try:
                                success = fut.result()
                                if success:
                                    total_saved += 1
                            except Exception:
                                pass
                    # after each page flush partial batch if large
                    flush_batch_to_csv()
                    print(f" Pg{page} done", end=' ')
                    # polite pause between pages
                    time.sleep(random.uniform(0.8, 1.5))
    except KeyboardInterrupt:
        print("\nInterrupted by user. Flushing remaining batch...")
    finally:
        # flush any remaining results
        flush_batch_to_csv(force=True)
        print(f"\nFinished. Total processed (attempts): {total_saved}")

if __name__ == "__main__":
    main()



Crawling parent 931 -> Thời trang nữ | subcats: 16

 Sub: Áo nữ (ID 1698)
+++++++++++++++++++++...++++++++++++++++ Pg1 done +++++++++++++++++++++++.++++++++++++++++ Pg2 done ++++++++++++++++++++++++++++++++++.+++++ Pg3 done 
 Sub: Đầm nữ (ID 941)
++++++++++++++++++++++++++++++++++++++++ Pg1 done ++++++++++++++++++++++++++++++++.+++++++ Pg2 done ++++++++++++++++++++++++++++++++++++++++ Pg3 done 
 Sub: Chân váy (ID 5404)
++++++++++++++++++++++++++++++++++++++++ Pg1 done ++++++++++++++++++++++++++++++++++++++++ Pg2 done ++++++++++++++++++++++++++++++.+++++++++ Pg3 done 
 Sub: Quần nữ (ID 27600)
++++++++++++++++++++++++++++++++++++++++ Pg1 done ++++++++++++++++++++++++++++++++++++++++ Pg2 done ++++++++++++++++++++++++.+.+++++++++++++ Pg3 done 
 Sub: Áo vest - Áo khoác nữ (ID 936)
++++++++++++++++++++++++++++++++++++++.+ Pg1 done ++++++++++++++++++++++++++++++++++++++++ Pg2 done ++++++++++++++++++++++++++++++++++++++++ Pg3 done 
 Sub: Áo liền quần - Bộ trang phục (ID 1702)
++++++++++++++++

In [12]:
import pandas as pd
import os

# ================= CẤU HÌNH =================
# Thay tên file csv của bạn vào đây (file mới nhất V23)
FILE_PATH = "tiki_Chi_3_pages.csv" 

# ================= XỬ LÝ =================
if os.path.exists(FILE_PATH):
    # Đọc file CSV
    try:
        df = pd.read_csv(FILE_PATH)
        
        print(f"✅ Đã tải file: {FILE_PATH}")
        print(f"📊 Tổng số sản phẩm: {len(df)}")
        print(f"gồm {len(df.columns)} cột dữ liệu.\n")
        print("-" * 50)

        # Tính toán dữ liệu thiếu
        missing_count = df.isnull().sum()
        missing_pct = (df.isnull().sum() / len(df)) * 100

        # Tạo bảng thống kê
        stats = pd.DataFrame({
            'Tên Cột': df.columns,
            'Số dòng thiếu': missing_count,
            'Tỉ lệ thiếu (%)': missing_pct.round(2) # Làm tròn 2 chữ số
        })

        # Sắp xếp để các cột thiếu nhiều nhất hiện lên đầu
        stats = stats.sort_values(by='Tỉ lệ thiếu (%)', ascending=False)

        # Hiển thị bảng
        # Nếu chạy trên Jupyter/Colab thì dùng display(stats), console thì dùng print
        print(stats.to_string(index=False))
        
        print("-" * 50)
        print("💡 NHẬN XÉT NHANH:")
        
        # Logic nhận xét tự động
        shop_missing = stats[stats['Tên Cột'] == 'cancel_by_seller_rate']['Tỉ lệ thiếu (%)'].values[0]
        if shop_missing > 50:
            print(f"⚠️ Cảnh báo: Dữ liệu Shop (Hủy/Hoàn) bị thiếu {shop_missing}% -> Có thể bị chặn hoặc Shop mới.")
        else:
            print(f"✅ Dữ liệu Shop khá ổn (chỉ thiếu {shop_missing}%).")
            
        discount_missing = stats[stats['Tên Cột'] == 'original_price']['Tỉ lệ thiếu (%)'].values[0]
        print(f"ℹ️ Original Price thiếu {discount_missing}%: Bình thường (Sản phẩm không giảm giá thì không có giá gốc).")

    except Exception as e:
        print(f"❌ Lỗi đọc file: {e}")
else:
    print(f"❌ Không tìm thấy file: {FILE_PATH}")

✅ Đã tải file: tiki_Chi_3_pages.csv
📊 Tổng số sản phẩm: 5057
gồm 29 cột dữ liệu.

--------------------------------------------------
                     Tên Cột  Số dòng thiếu  Tỉ lệ thiếu (%)
               quantity_sold           2006            39.67
                      origin            234             4.63
          return_rate_status             51             1.01
                 return_rate             51             1.01
cancel_by_seller_rate_status             51             1.01
       cancel_by_seller_rate             51             1.01
              total_follower             51             1.01
          store_review_count             51             1.01
                  store_name             51             1.01
                    store_id             51             1.01
                 image_count              0             0.00
                 is_official              0             0.00
                  brand_name              0             0.00
             